In [1]:
import os, re, zipfile, cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)


TensorFlow: 2.19.0


In [2]:
from google.colab import drive
drive.mount("/content/drive")

zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"
print("Dataset extracted to:", BASE)


Mounted at /content/drive
Dataset extracted to: /content/dataset/Parkinson's Dataset Final


In [3]:
records = []

def scan_images(root_dir, label):
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                records.append({
                    "path": os.path.join(root, f),
                    "label": label
                })

# New Hand PD
scan_images(os.path.join(BASE, "New hand pd Dataset", "Parkinson's"), 1)
scan_images(os.path.join(BASE, "New hand pd Dataset", "Healthy"), 0)

# Old Hand PD
scan_images(os.path.join(BASE, "Hand Pd", "Parkinson"), 1)
scan_images(os.path.join(BASE, "Hand Pd", "Healthy"), 0)

image_df = pd.DataFrame(records)

print("TOTAL IMAGES:", len(image_df))
print("\nClass distribution:")
print(image_df["label"].value_counts())


TOTAL IMAGES: 512

Class distribution:
label
1    279
0    233
Name: count, dtype: int64


In [4]:
OLD_HAND_PD = os.path.join(BASE, "Hand Pd")

for root, dirs, files in os.walk(OLD_HAND_PD):
    level = root.replace(OLD_HAND_PD, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:5]:
        print(f"{indent}    {f}")
    if level >= 3:
        break


Hand Pd/
    Spiral_HandPD/
        parkinson/
            0136-2.jpg
            0030-2.jpg
            0051-1.jpg
            0246-2.jpg
            0053-3.jpg
        healthy/
            0129-2.jpg
            0289-1.jpg
            0102-1.jpg
            0068-6.jpg
            0131-8.jpg
    Meander_HandPD/
        parkinson/
            0002-7.jpg
            0045-8.jpg
            0229-8.jpg
            0197-6.jpg
            0246-7.jpg
        healthy/
            0068-6.jpg
            0131-8.jpg
            0112-7.jpg
            ww.jpg
            0287-7.jpg


In [5]:
records = []

def scan_old_handpd(root_dir):
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                full_path = os.path.join(root, f)

                if "/healthy/" in full_path.lower():
                    label = 0
                elif "/parkinson/" in full_path.lower():
                    label = 1
                else:
                    continue   # safety

                records.append({
                    "path": full_path,
                    "label": label
                })

def scan_new_handpd(root_dir):
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                full_path = os.path.join(root, f)

                if "healthy" in full_path.lower():
                    label = 0
                else:
                    label = 1

                records.append({
                    "path": full_path,
                    "label": label
                })

# ---- RUN SCANS ----
scan_new_handpd(os.path.join(BASE, "New hand pd Dataset"))
scan_old_handpd(os.path.join(BASE, "Hand Pd"))

image_df = pd.DataFrame(records)

print("TOTAL IMAGES:", len(image_df))
print("\nClass distribution:")
print(image_df["label"].value_counts())


TOTAL IMAGES: 1320

Class distribution:
label
1    871
0    449
Name: count, dtype: int64


In [6]:
print("Old Hand PD images:",
      image_df["path"].str.contains("Hand Pd", case=False).sum())

image_df[image_df["path"].str.contains("Hand Pd", case=False)].head()


Old Hand PD images: 1320


,path,label
0,/content/dataset/Parkinson's Dataset Final/New...,1
1,/content/dataset/Parkinson's Dataset Final/New...,1
2,/content/dataset/Parkinson's Dataset Final/New...,1
3,/content/dataset/Parkinson's Dataset Final/New...,1
4,/content/dataset/Parkinson's Dataset Final/New...,1


In [7]:
image_df[image_df["path"].str.contains("Hand Pd", case=False)].sample(5)


,path,label
835,/content/dataset/Parkinson's Dataset Final/Han...,0
33,/content/dataset/Parkinson's Dataset Final/New...,1
79,/content/dataset/Parkinson's Dataset Final/New...,1
66,/content/dataset/Parkinson's Dataset Final/New...,1
324,/content/dataset/Parkinson's Dataset Final/New...,0


In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    image_df,
    test_size=0.30,
    stratify=image_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", train_df["label"].value_counts())
print("Val:", val_df["label"].value_counts())
print("Test:", test_df["label"].value_counts())


Train: label
1    610
0    314
Name: count, dtype: int64
Val: label
1    131
0     67
Name: count, dtype: int64
Test: label
1    130
0     68
Name: count, dtype: int64


In [9]:
import cv2
import numpy as np

IMG_SIZE = 224

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    return img[..., np.newaxis]


In [10]:
def make_dataset(df, batch_size=16, augment=False):
    paths = df["path"].values
    labels = df["label"].values

    def _load(path, label):
        img = tf.numpy_function(load_image, [path], tf.float32)
        img.set_shape((224,224,1))
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.shuffle(512).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [11]:
def augment_fn(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    return img, label


In [12]:
train_ds = make_dataset(train_df, augment=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)


In [13]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size=16, embed_dim=128):
        super().__init__()
        self.patch_size = patch_size
        self.proj = layers.Conv2D(
            filters=embed_dim,
            kernel_size=patch_size,
            strides=patch_size,
            padding="valid"
        )

    def call(self, x):
        x = self.proj(x)  # (B, H/ps, W/ps, embed_dim)
        x = tf.reshape(x, (tf.shape(x)[0], -1, x.shape[-1]))
        return x


In [18]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.embed = tf.keras.layers.Embedding(num_patches, dim)

    def call(self, x):
        positions = tf.range(tf.shape(x)[1])
        return x + self.embed(positions)


In [19]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, out_dim, num_heads=4):
        super().__init__()
        self.out_dim = out_dim

        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()

        self.attn = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=out_dim
        )

        self.mlp = tf.keras.layers.Dense(out_dim, activation="gelu")
        self.proj = None

    def build(self, input_shape):
        if input_shape[-1] != self.out_dim:
            self.proj = tf.keras.layers.Dense(self.out_dim)

    def call(self, x):
        shortcut = x
        if self.proj is not None:
            shortcut = self.proj(shortcut)

        x = self.norm1(shortcut + self.attn(x, x))
        x = self.norm2(x + self.mlp(x))
        return x


In [20]:
def build_custom_vit_feature_extractor():
    inputs = tf.keras.Input(shape=(224, 224, 1))

    # Patch embedding
    x = PatchEmbedding(patch_size=16, embed_dim=526)(inputs)

    # Positional encoding
    x = PositionalEncoding(num_patches=196, dim=526)(x)

    # YOUR hierarchical ViT (UNCHANGED)
    for dim in [526, 256, 128, 64, 32, 16, 8, 4]:
        x = TransformerBlock(out_dim=dim)(x)

    # Token pooling → FEATURE VECTOR
    features = tf.keras.layers.GlobalAveragePooling1D()(x)

    return tf.keras.Model(
        inputs,
        features,
        name="Custom_ViT_Feature_Extractor"
    )


In [21]:
vit_extractor = build_custom_vit_feature_extractor()
vit_extractor.summary()


ValueError: Exception encountered when calling TransformerBlock.call().

[1mCould not automatically infer the output shape / dtype of 'transformer_block_7' (of type TransformerBlock). Either the `TransformerBlock.call()` method is incorrect, or you need to implement the `TransformerBlock.compute_output_spec() / compute_output_shape()` method. Error encountered:

Dimensions must be equal, but are 256 and 526 for '{{node add}} = AddV2[T=DT_FLOAT](dense_24_1/BiasAdd, multi_head_attention_7_1/attention_output_1/BiasAdd)' with input shapes: [?,?,256], [?,?,526].[0m

Arguments received by TransformerBlock.call():
  • args=('<KerasTensor shape=(None, None, 526), dtype=float32, sparse=False, ragged=False, name=keras_tensor_52>',)
  • kwargs=<class 'inspect._empty'>